# Практика · 08. Пулінг і зменшення розмірності> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)Що зробимо:1. напишемо **власний max-pooling на numpy** і звіримо його з `nn.MaxPool2d` через `np.allclose`;2. порахуємо форму карти після пулінгу **руками** й перевіримо проти фактичного `.shape`;3. переконаємось, що в пулінгу **нуль параметрів**, а в згортки з кроком — ні;4. зробимо чотири заміри: терпимість до зсуву з пулінгом і без, максимум проти   середнього, пулінг проти згортки з кроком, розпрямлення проти глобального усереднення;5. подивимось, де пулінг **псує** ознаку — на дрібному предметі й на текстурі.**Мережа не потрібна:** датасет ми малюємо самі, формулами.⚠️ Зошит навчає шість маленьких мереж. На звичайному ноутбуці це кілька хвилин.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn

# зерна фіксуємо один раз на весь зошит — інакше числа не збігатимуться з лекцією
SEED = 42
torch.manual_seed(SEED)
torch.set_num_threads(4)

print("torch", torch.__version__, "· numpy", np.__version__)
print("зерно", SEED)

## 1 · Датасет: фігури 28×28Три класи — коло, квадрат, ромб. Центр фігури зсунуто не більше ніж на `shift` пікселів,зверху накладено невеликий шум. Той самий генератор, що в решті блоку.

In [ ]:
def draw_one(kind, center_x, center_y, radius, rng, noise=0.08):
    """Одна фігура 28×28 у відтінках сірого 0..1."""
    yy, xx = np.mgrid[0:28, 0:28]
    dx = xx - center_x
    dy = yy - center_y
    if kind == 0:                                    # коло
        mask = (dx * dx + dy * dy) <= radius * radius
    elif kind == 1:                                  # квадрат
        mask = (np.abs(dx) <= radius) & (np.abs(dy) <= radius)
    else:                                            # ромб
        mask = (np.abs(dx) + np.abs(dy)) <= radius
    image = mask.astype(np.float32)
    # шум потрібен, щоб задача не розвʼязувалась одним порогом
    image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0.0, 1.0)


def make_dataset(n, shift, seed):
    """n картинок; центр кожної зсунуто не більше ніж на shift пікселів."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n, 1, 28, 28), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        kind = i % 3
        center_x = 13.5 + rng.integers(-shift, shift + 1)
        center_y = 13.5 + rng.integers(-shift, shift + 1)
        radius = rng.integers(5, 8)
        X[i, 0] = draw_one(kind, center_x, center_y, radius, rng)
        y[i] = kind
    return X, y


start = time.time()
X_train, y_train = make_dataset(1200, shift=1, seed=42)
# перевірочні набори: той самий розподіл і зсуви від 0 до 6 пікселів
shifted_sets = {s: make_dataset(600, shift=s, seed=100 + s) for s in range(7)}
print(f"навчальна вибірка {X_train.shape}, згенеровано за {time.time() - start:.1f} с")
print("класів:", np.bincount(y_train), "· набори для перевірки:", list(shifted_sets))

## 2 · Власний max-pooling на numpyПулінг — це ковзне вікно без ваг. Напишемо його найпрямішим способом: два цикли попозиціях вікна, а всередині — максимум по двох останніх осях. Ніякої магії тут немає,і зараз ми це доведемо звіркою з бібліотекою.

In [ ]:
def max_pool_numpy(x, k=2, stride=2):
    """Максимум у вікні k×k із кроком stride. x має форму (N, C, H, W)."""
    n, c, h, w = x.shape
    out_h = (h - k) // stride + 1
    out_w = (w - k) // stride + 1
    out = np.zeros((n, c, out_h, out_w), dtype=x.dtype)
    for i in range(out_h):
        for j in range(out_w):
            # вікно беремо зрізом, максимум — по висоті й ширині вікна
            window = x[:, :, i * stride:i * stride + k, j * stride:j * stride + k]
            out[:, :, i, j] = window.max(axis=(2, 3))
    return out


our_result = max_pool_numpy(X_train[:16])
torch_result = nn.MaxPool2d(2)(torch.from_numpy(X_train[:16])).numpy()

assert np.allclose(our_result, torch_result), "розрахунок розійшовся!"
print("наш max-pool:  ", our_result.shape)
print("nn.MaxPool2d:  ", torch_result.shape)
print("✅ збігається до останнього знака")

Те саме для середнього. Різниця в одному рядку: замість `max` беремо `mean`.

In [ ]:
def avg_pool_numpy(x, k=2, stride=2):
    """Середнє у вікні k×k із кроком stride."""
    n, c, h, w = x.shape
    out_h = (h - k) // stride + 1
    out_w = (w - k) // stride + 1
    out = np.zeros((n, c, out_h, out_w), dtype=x.dtype)
    for i in range(out_h):
        for j in range(out_w):
            window = x[:, :, i * stride:i * stride + k, j * stride:j * stride + k]
            out[:, :, i, j] = window.mean(axis=(2, 3))
    return out


our_avg = avg_pool_numpy(X_train[:16])
torch_avg = nn.AvgPool2d(2)(torch.from_numpy(X_train[:16])).numpy()
assert np.allclose(our_avg, torch_avg, atol=1e-6), "розрахунок розійшовся!"
print("✅ середнє теж збігається")
print("максимум завжди не менший за середнє:",
      bool(np.all(our_result >= our_avg)))

## 3 · Форма після пулінгу: рахуємо рукамиФормула та сама, що для згортки:    розмір виходу = (розмір входу − k) // stride + 1Найцікавіший рядок — 7 із вікном 2: виходить 3, а не 3.5. Останній стовпчик простоне потрапляє в жодне вікно й **зникає мовчки**.

In [ ]:
def expected_side(side, k, stride):
    """Скільки клітинок буде на виході — рахуємо формулою, без запуску шару."""
    return (side - k) // stride + 1


print(f"{'вхід':>6} {'k':>3} {'stride':>7} {'формула':>9} {'.shape':>8}  збіг")
for side, k, stride in [(28, 2, 2), (14, 2, 2), (7, 2, 2), (28, 3, 2), (28, 2, 1), (5, 2, 2)]:
    by_hand = expected_side(side, k, stride)
    probe = torch.zeros(1, 1, side, side)
    actual = nn.MaxPool2d(k, stride=stride)(probe).shape[-1]
    assert by_hand == actual, f"розійшлося на {side}/{k}/{stride}"
    print(f"{side:>6} {k:>3} {stride:>7} {by_hand:>9} {actual:>8}  ✅")

print("\nвтрачений стовпчик: 7 // 2 дає 3 вікна, восьмий рядок узяти нізвідки")

## 4 · У пулінгу нуль параметрівПорівняємо три способи зменшити карту з 8 каналів удвічі. Параметри рахуємо не зпамʼяті, а через `sum(p.numel() for p in layer.parameters())`.

In [ ]:
def count_parameters(module):
    """Скільки чисел цей шар навчатиме."""
    return sum(p.numel() for p in module.parameters())


ways_to_shrink = {
    "nn.MaxPool2d(2)": nn.MaxPool2d(2),
    "nn.AvgPool2d(2)": nn.AvgPool2d(2),
    "nn.Conv2d(8, 8, 3, stride=2)": nn.Conv2d(8, 8, 3, stride=2, padding=1),
}
for name, layer in ways_to_shrink.items():
    print(f"{name:<30} параметрів: {count_parameters(layer)}")

assert count_parameters(nn.MaxPool2d(2)) == 0, "у пулінгу не має бути параметрів"
print("\n✅ у пулінгу рівно нуль параметрів, а згортка з кроком коштує 584 ваги")

## 5 · Скільки множень коштує кожен спосібСекунди залежать від того, чим ще зайнята машина, а множення — ні. Порахуємо їх точно:для згортки це «клітинок виходу × вхідних каналів × площа ядра».

In [ ]:
def conv_multiplications(in_ch, out_ch, side_out, k=3):
    """Множень на одну картинку в одному згортковому шарі."""
    return out_ch * side_out * side_out * in_ch * k * k


stacks = {
    "без пулінгу":        conv_multiplications(1, 8, 28) + conv_multiplications(8, 16, 28) + 12544 * 3,
    "згортка + MaxPool":  conv_multiplications(1, 8, 28) + conv_multiplications(8, 16, 14) + 784 * 3,
    "крок у згортці":     conv_multiplications(1, 8, 14) + conv_multiplications(8, 16, 7) + 784 * 3,
    "окремий шар-крок":   (conv_multiplications(1, 8, 28) + conv_multiplications(8, 8, 14)
                           + conv_multiplications(8, 16, 14) + conv_multiplications(16, 16, 7)
                           + 784 * 3),
}
for name, mults in stacks.items():
    print(f"{name:<20} {mults:>10,} множень".replace(",", " "))

print("\nпулінг проти кроку в згортці:",
      round(stacks["згортка + MaxPool"] / stacks["крок у згортці"], 1), "раза дорожче")

## 6 · Шість мереж, які ми порівнюватимемоУсі мають однакову глибину — два згорткові шари з 8 і 16 ядрами. Відрізняється тільките, чим (і чи взагалі) зменшується карта, і яка голова стоїть у кінці.

In [ ]:
def make_model(kind):
    """Один стос за іменем. Зерно перед створенням фіксуємо, щоб початкові
    ваги в усіх мереж були з того самого генератора."""
    torch.manual_seed(0)
    if kind == "без пулінгу":
        return nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(16 * 28 * 28, 3))
    if kind == "MaxPool":
        return nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(16 * 7 * 7, 3))
    if kind == "AvgPool":
        return nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.AvgPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AvgPool2d(2),
            nn.Flatten(), nn.Linear(16 * 7 * 7, 3))
    if kind == "крок у згортці":
        return nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(16 * 7 * 7, 3))
    if kind == "окремий шар-крок":
        return nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.Conv2d(8, 8, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(16 * 7 * 7, 3))
    # глобальне усереднення замість розпрямлення
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 3))


for kind in ["без пулінгу", "MaxPool", "AvgPool", "крок у згортці",
             "окремий шар-крок", "усереднення"]:
    model = make_model(kind)
    head = [m for m in model if isinstance(m, nn.Linear)][0]
    print(f"{kind:<18} усього {count_parameters(model):>6} · у голові {count_parameters(head):>6}")

Навчання й перевірка — дві короткі функції. Порядок прикладів у кожній епосі тежбереться з генератора із зафіксованим зерном, інакше числа попливуть.

In [ ]:
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)


def train_model(model, epochs=12, batch=64):
    """Навчаємо й повертаємо витрачений час у секундах."""
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    order = torch.Generator().manual_seed(0)
    start = time.time()
    for _ in range(epochs):
        permutation = torch.randperm(len(X_train_t), generator=order)
        for i in range(0, len(permutation), batch):
            index = permutation[i:i + batch]
            optimizer.zero_grad()
            loss = loss_fn(model(X_train_t[index]), y_train_t[index])
            loss.backward()
            optimizer.step()
    return time.time() - start


def accuracy(model, X, y):
    """Частка правильних відповідей на готовому наборі."""
    model.eval()
    with torch.no_grad():
        predicted = model(torch.from_numpy(X)).argmax(1).numpy()
    model.train()
    return float((predicted == y).mean())


def train_and_measure(kind):
    """Навчає стос і міряє точність на всіх зсувах від 0 до 6."""
    model = make_model(kind)
    seconds = train_model(model)
    by_shift = {s: round(accuracy(model, *shifted_sets[s]), 3) for s in range(7)}
    print(f"{kind:<18} {seconds:>6.1f} с · " +
          " ".join(f"±{s}:{by_shift[s]:.3f}" for s in range(7)))
    return {"model": model, "seconds": seconds, "acc": by_shift,
            "params": count_parameters(model)}


results = {}
print("навчаємо (це найдовша частина зошита)\n")

## 7 · Замір 1: терпимість до зсуву з пулінгом і безОбидві мережі бачать під час навчання лише фігури майже по центру (зсув ±1). Перевіряємоїх на фігурах, зсунутих усе далі. Якщо пулінг справді додає терпимість до зсуву — це маєбути видно на правому кінці рядка.

In [ ]:
results["без пулінгу"] = train_and_measure("без пулінгу")
results["MaxPool"] = train_and_measure("MaxPool")

print()
for shift in (1, 3, 6):
    without = results["без пулінгу"]["acc"][shift]
    with_pool = results["MaxPool"]["acc"][shift]
    print(f"зсув ±{shift}: без пулінгу {without:.3f} · з пулінгом {with_pool:.3f} "
          f"· різниця {with_pool - without:+.3f}")
print(f"\nпараметрів: без пулінгу {results['без пулінгу']['params']}, "
      f"з пулінгом {results['MaxPool']['params']} "
      f"(у {results['без пулінгу']['params'] / results['MaxPool']['params']:.1f} раза менше)")
print("рівень вгадування при трьох класах: 0.333")

## 8 · Замір 2: максимум проти середньогоТа сама мережа, змінено одне слово: `MaxPool2d` на `AvgPool2d`. Параметрів в обоходнаково — нуль на пулінг.

In [ ]:
results["AvgPool"] = train_and_measure("AvgPool")

print()
for shift in (1, 3, 6):
    print(f"зсув ±{shift}: max {results['MaxPool']['acc'][shift]:.3f} "
          f"· avg {results['AvgPool']['acc'][shift]:.3f}")
print("\nмаксимум не змінюється, поки ознака рухається всередині вікна;")
print("середнє враховує всі чотири числа, тож реагує на будь-який рух.")

## 9 · Замір 3: пулінг проти згортки з крокомДва способи зменшити карту навчаними вагами. Перший згортає крок у саму згортку(параметрів не додає), другий ставить окремий шар-зменшувач (додає).

In [ ]:
results["крок у згортці"] = train_and_measure("крок у згортці")
results["окремий шар-крок"] = train_and_measure("окремий шар-крок")

print()
print(f"{'спосіб':<18} {'параметрів':>11} {'множень':>10} {'±3':>7} {'±6':>7}")
for kind, mults in [("MaxPool", stacks["згортка + MaxPool"]),
                    ("крок у згортці", stacks["крок у згортці"]),
                    ("окремий шар-крок", stacks["окремий шар-крок"])]:
    row = results[kind]
    print(f"{kind:<18} {row['params']:>11} {mults:>10} "
          f"{row['acc'][3]:>7.3f} {row['acc'][6]:>7.3f}")

## 10 · Замір 4: розпрямлення проти глобального усередненняТіло мережі те саме, що в рядку `MaxPool`. Змінюємо лише голову: замість `Flatten` +`Linear(784, 3)` ставимо `AdaptiveAvgPool2d(1)` + `Linear(16, 3)`.

In [ ]:
results["усереднення"] = train_and_measure("усереднення")

flatten_head = count_parameters([m for m in results["MaxPool"]["model"]
                                 if isinstance(m, nn.Linear)][0])
gap_head = count_parameters([m for m in results["усереднення"]["model"]
                             if isinstance(m, nn.Linear)][0])
print()
print(f"ваг у голові: Flatten {flatten_head} · усереднення {gap_head} "
      f"(у {flatten_head / gap_head:.0f} разів менше)")
print(f"усього параметрів: {results['MaxPool']['params']} проти {results['усереднення']['params']}")
print()
for shift in range(7):
    print(f"зсув ±{shift}: Flatten {results['MaxPool']['acc'][shift]:.3f} "
          f"· усереднення {results['усереднення']['acc'][shift]:.3f}")

Подивись на праву колонку згори вниз: у мережі з глобальним усередненням вона майжене спадає. Причина проста — середнє по всій площі не залежить від того, у якому місціплощі лежали числа. Це пулінг, доведений до краю, а разом із ним до краю доведена йтерпимість до зсуву.Ціна теж видна: на тому розподілі, на якому мережа вчилася (зсув ±1), коротка головатрохи програє довгій.

In [ ]:
# розмір входу: Flatten чекає рівно 784 числа, усереднення — будь-яку карту
for side in (28, 40, 56):
    probe = torch.zeros(1, 1, side, side)
    body = nn.Sequential(*list(results["MaxPool"]["model"])[:6])
    feature_map = body(probe)
    try:
        results["MaxPool"]["model"](probe)
        flatten_ok = "рахує"
    except RuntimeError:
        flatten_ok = "RuntimeError"
    results["усереднення"]["model"](probe)
    print(f"вхід {side}×{side} → карта {tuple(feature_map.shape[1:])} "
          f"({feature_map.numel()} чисел) · Flatten: {flatten_ok} · усереднення: рахує")

## 11 · Де пулінг псує ознакуТри короткі досліди, у яких пулінг втрачає саме те, заради чого ознаку й будували.**Дрібний предмет.** Світла точка 2×2 після трьох шарів середнього пулінгуусереднюється з площею 8×8.

In [ ]:
small_object = np.zeros((1, 1, 28, 28), dtype=np.float32)
small_object[0, 0, 12:14, 12:14] = 1.0            # яскрава точка 2×2

after_avg = small_object.copy()
after_max = small_object.copy()
for _ in range(3):
    after_avg = avg_pool_numpy(after_avg)
    after_max = max_pool_numpy(after_max)

print("яскравість точки на вході:", small_object.max())
print("після трьох avg-пулінгів: ", round(float(after_avg.max()), 4),
      f"(ослабла у {1 / after_avg.max():.0f} разів)")
print("після трьох max-пулінгів: ", round(float(after_max.max()), 4))
print("\nале координата після трьох зменшень відома з точністю до 8 пікселів:")
print("карта стала", after_avg.shape[-1], "×", after_avg.shape[-1])

**Густина текстури.** Дві ділянки з різною часткою світлих пікселів. Максимум відповідаєна питання «чи є», тому насичується; середнє відповідає «скільки», тому густину зберігає.

In [ ]:
rng = np.random.default_rng(7)
sparse = (rng.random((1, 1, 32, 32)) < 0.05).astype(np.float32)   # 5 % площі
dense = (rng.random((1, 1, 32, 32)) < 0.20).astype(np.float32)    # 20 % площі

print(f"{'шар':>4} {'max рідка':>11} {'max густа':>11} {'avg рідка':>11} {'avg густа':>11}")
sparse_max, dense_max = sparse.copy(), dense.copy()
sparse_avg, dense_avg = sparse.copy(), dense.copy()
print(f"{0:>4} {sparse_max.mean():>11.3f} {dense_max.mean():>11.3f} "
      f"{sparse_avg.mean():>11.3f} {dense_avg.mean():>11.3f}")
for layer in (1, 2, 3):
    sparse_max, dense_max = max_pool_numpy(sparse_max), max_pool_numpy(dense_max)
    sparse_avg, dense_avg = avg_pool_numpy(sparse_avg), avg_pool_numpy(dense_avg)
    print(f"{layer:>4} {sparse_max.mean():>11.3f} {dense_max.mean():>11.3f} "
          f"{sparse_avg.mean():>11.3f} {dense_avg.mean():>11.3f}")

print("\nмаксимум зводить обидві текстури до одиниці — різниця густин зникає;")
print("середнє тримає 0.05 і 0.20 на будь-якій глибині.")

**Тонка структура.** А тепер дзеркальна сліпота середнього: смужки шириною в одинпіксель і рівна сіра заливка тієї самої яскравості.

In [ ]:
stripes = np.zeros((1, 1, 16, 16), dtype=np.float32)
stripes[0, 0, :, 1::2] = 1.0                       # 0 1 0 1 0 1 ...
flat_gray = np.full((1, 1, 16, 16), 0.5, dtype=np.float32)

stripes_avg = avg_pool_numpy(stripes)
gray_avg = avg_pool_numpy(flat_gray)

print("вхід різний:      ", not np.allclose(stripes, flat_gray))
print("після avg-пулінгу:", "однакові" if np.allclose(stripes_avg, gray_avg) else "різні")
print("значення в обох випадках:", float(stripes_avg[0, 0, 0, 0]))
print("\nа максимум?", float(max_pool_numpy(stripes)[0, 0, 0, 0]),
      "проти", float(max_pool_numpy(flat_gray)[0, 0, 0, 0]), "— ці він розрізняє")

## Підсумок замірівУсе, що ми зміряли, одним табло.

In [ ]:
print(f"{'стос':<18} {'параметрів':>11} {'±1':>7} {'±3':>7} {'±6':>7}")
for kind in ["без пулінгу", "MaxPool", "AvgPool", "крок у згортці",
             "окремий шар-крок", "усереднення"]:
    row = results[kind]
    print(f"{kind:<18} {row['params']:>11} {row['acc'][1]:>7.3f} "
          f"{row['acc'][3]:>7.3f} {row['acc'][6]:>7.3f}")

total_seconds = sum(row["seconds"] for row in results.values())
print(f"\nусього навчання: {total_seconds:.0f} с")

## Завдання### 🟢 Рівень 1Заміни у стосі `MaxPool` вікно 2×2 на 3×3 із кроком 3. Порахуй руками, якою стане картапісля двох таких шарів, перевір формулою й `.shape`, потім навчи мережу та порівняйточність на зсуві ±6 із числом із заміру 1.### 🟡 Рівень 2Додай **третій** блок «згортка + пулінг» і подивись, що станеться з точністю на зсунутихпредметах і зі зростанням рецептивного поля. Порахуй RF наскрізь за формулами з розділу 07лекції й перевір здогад числом.### 🔴 Рівень 3Напиши власний пулінг із **прямим і зворотним проходом**: максимум має пропускати градієнтлише в те число, яке перемогло. Звір свій градієнт із тим, що дає PyTorch(`tensor.grad` після `backward()`), через `np.allclose`.Докладніше — у [homework.html](homework.html).